# The Gauss–Markov model

The line fitted so far was a geometric fact about a table. Inference needs a probability model for how that table was produced. The Gauss–Markov theorem says that, under a short list of assumptions, least squares is the best linear unbiased estimator.


## Assumptions and unbiasedness

Conditional on a fixed full-rank design $X$, assume

$$
y = X\beta + \varepsilon, \qquad E(\varepsilon) = 0, \qquad \operatorname{Var}(\varepsilon) = \sigma^2 I.
$$

The errors have mean zero, a common variance, and no correlation with each other. Nothing yet says that they are normal. Normality will be used only when a $t$ or $F$ reference distribution is required.

The least-squares map is linear in $y$:

$$
\widehat\beta = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y = \beta + (X^{\mathsf T}X)^{-1}X^{\mathsf T}\varepsilon.
$$

Take the expectation and use $E(\varepsilon) = 0$:

$$
E(\widehat\beta) = \beta.
$$

The estimator is unbiased for every sample size at which $X$ has full rank. Unbiasedness does not say that a particular sample's $\widehat\beta$ equals $\beta$. It says the sampling average equals $\beta$.

The variance follows from $\operatorname{Var}(Ay) = A\operatorname{Var}(y)A^{\mathsf T}$ with $A = (X^{\mathsf T}X)^{-1}X^{\mathsf T}$:

$$
\operatorname{Var}(\widehat\beta) = \sigma^2 (X^{\mathsf T}X)^{-1}.
$$

Among all estimators that are linear in $y$ and unbiased for $\beta$, the Gauss–Markov theorem states that this estimator has the smallest variance on every linear functional $c^{\mathsf T}\beta$. The proof compares an arbitrary competitor $Cy$ with $CX = I$ to the least-squares map and shows that the difference of the two variance matrices is positive semidefinite. The theorem does not claim that a nonlinear estimator, or an estimator that is allowed to be biased, cannot do better.


## Variances on the five-point design

For simple regression with an intercept, the inverse of $X^{\mathsf T}X$ has a known diagonal. With $\sigma^2$ treated as known,

$$
\operatorname{Var}(b_1) = \dfrac{\sigma^2}{S_{xx}}, \qquad
\operatorname{Var}(b_0) = \sigma^2\left(\dfrac{1}{n} + \dfrac{\bar x^2}{S_{xx}}\right).
$$

On $x = 1, 2, 3, 4, 5$, one has $n = 5$, $\bar x = 3$, and $S_{xx} = 10$. If $\sigma^2 = 1$,

$$
\operatorname{Var}(b_1) = \dfrac{1}{10}, \qquad
\operatorname{Var}(b_0) = \dfrac{1}{5} + \dfrac{9}{10} = \dfrac{11}{10}.
$$

The slope is estimated more precisely when the $x$ values are more spread out, because $S_{xx}$ stands in the denominator. The intercept is estimated more precisely when $\bar x$ is near zero, because the fit does not have to travel a long distance from the center of the data back to $x = 0$.


In [1]:
# Read the slope and intercept variances off the inverse of X'X.
import numpy as np
x = np.array([1.0, 2, 3, 4, 5])
X = np.column_stack([np.ones(5), x])
sigma2 = 1.0
covariance = sigma2 * np.linalg.inv(X.T @ X)
print("Var(b0), Var(b1)")
print(np.diag(covariance))
assert abs(np.diag(covariance)[1] - 0.1) < 1e-12
assert abs(np.diag(covariance)[0] - 1.1) < 1e-12


Var(b0), Var(b1)
[1.1 0.1]


## A sampling check

Draw $8{,}000$ independent samples from the model that the algebra describes: $x$ stays fixed at $1, 2, 3, 4, 5$, the true line is $1 + 2x$, and the errors are independent standard normal, so $\sigma^2 = 1$. For each sample compute $b_1 = S_{xy}/S_{xx}$. Unbiasedness predicts that the average of these slopes is $2$. The variance formula predicts that their sample variance is $1/10$.

This experiment does not prove the theorem. It checks that the random mechanism in the code is the mechanism the theorem is about.


In [2]:
# The average fitted slope should sit near 2, and its variance near 1/10.
import numpy as np
rng = np.random.default_rng(0)
x = np.array([1.0, 2, 3, 4, 5])
xbar = x.mean()
sxx = np.sum((x - xbar) ** 2)
noise = rng.normal(0.0, 1.0, size=(8000, 5))
y = 1 + 2 * x + noise
ybar = y.mean(axis=1)
sxy = (y - ybar[:, None]) @ (x - xbar)
slope = sxy / sxx
print("mean of b1", slope.mean())
print("variance of b1", slope.var(ddof=1))
print("hand variance", 1 / sxx)
assert abs(slope.mean() - 2) < 0.02
assert abs(slope.var(ddof=1) - 0.1) < 0.015


mean of b1 2.003568328693286
variance of b1 0.09803110089441196
hand variance 0.1


## Pitfall

If the errors have unequal variances, $\operatorname{Var}(\varepsilon) = \sigma^2 I$ is false and ordinary least squares remains unbiased but is no longer the best linear unbiased estimator. A one-line illustration of the variance assumption failing is enough here; weighted least squares is derived in its own lesson. Suppose the five errors had variances $1, 1, 1, 1, 25$ instead of $1, 1, 1, 1, 1$. The ordinary formula $\sigma^2/S_{xx}$ is then the wrong number, because there is no single $\sigma^2$ to insert.


In [3]:
# Heteroscedastic variances have no single sigma^2 for the ordinary formula.
variances = [1, 1, 1, 1, 25]
print("error variances", variances)
print("they are equal", len(set(variances)) == 1)
assert len(set(variances)) != 1


error variances [1, 1, 1, 1, 25]
they are equal False


## Summary

- Under $E(\varepsilon)=0$ and a fixed full-rank $X$, $E(\widehat\beta)=\beta$.
- Under the further assumption $\operatorname{Var}(\varepsilon)=\sigma^2 I$, $\operatorname{Var}(\widehat\beta)=\sigma^2(X^{\mathsf T}X)^{-1}$.
- On this design with $\sigma^2=1$, $\operatorname{Var}(b_1)=1/10$ and $\operatorname{Var}(b_0)=11/10$. A simulation with known normal errors reproduces those two numbers.
